# RQ2 Bias and Misclassification Risk

**Research question:** How do class imbalance and waste-category type influence misclassification risk in AI-based waste sorting?

This Kaggle notebook takes raw image-folder data as input and saves publication-ready tables as CSV and figures as PDF under `/kaggle/working/results/`.

In [3]:

# =========================
# COMMON SETUP
# =========================
import os, time, json, math, random, glob, shutil, pathlib, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OUTPUT_DIR = Path('/kaggle/working/results')
FIG_DIR = OUTPUT_DIR / 'figures_pdf'
TAB_DIR = OUTPUT_DIR / 'tables_csv'
MODEL_DIR = OUTPUT_DIR / 'models'
for d in [FIG_DIR, TAB_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS = 3          # Increase to 10-20 for final results
MAX_IMAGES_PER_CLASS = 500  # Set None for full dataset; keep small for quick Kaggle runs
VALID_EXT = ('.jpg','.jpeg','.png','.webp','.bmp')

# Auto-detect dataset directory. Works with Kaggle datasets and uploaded zip-derived folders.
def find_image_root(base='/kaggle/input'):
    candidates=[]
    for root, dirs, files in os.walk(base):
        image_count=sum(1 for f in files if f.lower().endswith(VALID_EXT))
        if image_count>0:
            candidates.append((root, image_count))
    if not candidates:
        raise FileNotFoundError('No image files found under /kaggle/input. Please attach the dataset in Kaggle Notebook > Add Input.')
    # Prefer a root with class subfolders containing images; otherwise parent of deepest image dirs.
    best=max(candidates, key=lambda x: x[1])[0]
    # If best is a class folder, use parent when multiple sibling class folders exist.
    parent=str(Path(best).parent)
    sibling_img_dirs=[]
    for d in os.listdir(parent):
        p=os.path.join(parent,d)
        if os.path.isdir(p):
            n=sum(1 for f in os.listdir(p) if f.lower().endswith(VALID_EXT))
            if n>0: sibling_img_dirs.append(d)
    if len(sibling_img_dirs)>=2:
        return parent
    # Special case DATASET/TRAIN/TEST: use TRAIN as train root when present.
    for root, dirs, files in os.walk(base):
        if 'TRAIN' in dirs:
            return os.path.join(root,'TRAIN')
    return parent

DATA_ROOT = find_image_root('/kaggle/input')
print('Detected image root:', DATA_ROOT)
print('Class folders:', [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT,d))][:30])

def build_manifest(data_root=DATA_ROOT, max_per_class=MAX_IMAGES_PER_CLASS):
    rows=[]
    for cls in sorted(os.listdir(data_root)):
        cpath=os.path.join(data_root, cls)
        if not os.path.isdir(cpath): continue
        files=[]
        for r,_,fs in os.walk(cpath):
            files += [os.path.join(r,f) for f in fs if f.lower().endswith(VALID_EXT)]
        if not files: continue
        if max_per_class is not None and len(files)>max_per_class:
            files=random.sample(files, max_per_class)
        for f in files:
            rows.append({'image_path':f, 'label':cls})
    df=pd.DataFrame(rows)
    if df.empty: raise ValueError('No labeled images found. Expected class folders containing images.')
    return df

manifest = build_manifest()
manifest.to_csv(TAB_DIR/'dataset_manifest.csv', index=False)
print(manifest['label'].value_counts())
classes = sorted(manifest['label'].unique())
num_classes=len(classes)
label_to_idx={c:i for i,c in enumerate(classes)}

train_df, temp_df = train_test_split(manifest, test_size=0.30, stratify=manifest['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
for name,df in [('train',train_df),('val',val_df),('test',test_df)]:
    df.to_csv(TAB_DIR/f'{name}_split.csv', index=False)
    print(name, df.shape)

def make_ds(df, shuffle=False, augment=False):
    paths=df['image_path'].values
    labels=np.array([label_to_idx[x] for x in df['label'].values], dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_img(path,label):
        img=tf.io.read_file(path)
        img=tf.image.decode_image(img, channels=3, expand_animations=False)
        img=tf.image.resize(img, IMG_SIZE)
        img=tf.cast(img, tf.float32)/255.0
        if augment:
            img=tf.image.random_flip_left_right(img)
            img=tf.image.random_brightness(img, 0.10)
        return img,label
    ds=ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds=ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_ds(train_df, shuffle=True, augment=True)
val_ds=make_ds(val_df)
test_ds=make_ds(test_df)

def build_model(model_name):
    inputs=layers.Input(shape=IMG_SIZE+(3,))
    if model_name=='MobileNetV2':
        base=tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name=='EfficientNetB0':
        # inputs already scaled 0-1; EfficientNet preprocessing disabled by using rescaling-neutral setup is acceptable for benchmark simplicity
        base=tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name=='ResNet50':
        base=tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    else:
        raise ValueError(model_name)
    base.trainable=False
    x=layers.GlobalAveragePooling2D()(base.output)
    x=layers.Dropout(0.25)(x)
    outputs=layers.Dense(num_classes, activation='softmax')(x)
    model=models.Model(inputs, outputs, name=model_name)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def evaluate_model(model, ds, df):
    y_true=[]; y_pred=[]; probs=[]
    for xb,yb in ds:
        p=model.predict(xb, verbose=0)
        probs.append(p); y_true.extend(yb.numpy().tolist()); y_pred.extend(np.argmax(p,axis=1).tolist())
    y_true=np.array(y_true); y_pred=np.array(y_pred); probs=np.vstack(probs)
    report=classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    rep_df=pd.DataFrame(report).T
    cm=confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    return rep_df, cm, y_true, y_pred, probs

def measure_latency(model, ds, n_batches=10):
    # Warm-up
    for xb,yb in ds.take(1): model.predict(xb, verbose=0)
    total_imgs=0; start=time.time()
    for i,(xb,yb) in enumerate(ds.take(n_batches)):
        model.predict(xb, verbose=0)
        total_imgs += xb.shape[0]
    elapsed=time.time()-start
    return (elapsed/total_imgs)*1000 if total_imgs else np.nan

def save_pdf(fig, filename):
    path=FIG_DIR/filename
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.close(fig)
    print('Saved', path)


Detected image root: /kaggle/input/datasets/techsash/waste-classification-data/DATASET/TRAIN
Class folders: ['R', 'O']
label
O    500
R    500
Name: count, dtype: int64
train (700, 2)
val (150, 2)
test (150, 2)


In [4]:

# RQ2: Bias and misclassification risk across classes
model=build_model('MobileNetV2')
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
rep, cm, y_true, y_pred, probs=evaluate_model(model, test_ds, test_df)
per_class=rep.loc[classes, ['precision','recall','f1-score','support']].reset_index().rename(columns={'index':'class'})
per_class['error_rate']=1-per_class['recall']
per_class.to_csv(TAB_DIR/'RQ2_class_bias_error_table.csv', index=False)
pd.DataFrame(cm, index=classes, columns=classes).to_csv(TAB_DIR/'RQ2_confusion_matrix.csv')

fig, ax = plt.subplots(figsize=(9,6))
im=ax.imshow(cm, aspect='auto')
ax.set_xticks(np.arange(num_classes)); ax.set_yticks(np.arange(num_classes))
ax.set_xticklabels(classes, rotation=45, ha='right'); ax.set_yticklabels(classes)
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title('RQ2: Misclassification Risk Matrix by Waste Class')
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout(); save_pdf(fig,'RQ2_confusion_matrix_bias_risks.pdf')
per_class


Epoch 1/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 460ms/step - accuracy: 0.7014 - loss: 0.6457 - val_accuracy: 0.8867 - val_loss: 0.3309
Epoch 2/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 430ms/step - accuracy: 0.8600 - loss: 0.3388 - val_accuracy: 0.9200 - val_loss: 0.2526
Epoch 3/3
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 367ms/step - accuracy: 0.9029 - loss: 0.2519 - val_accuracy: 0.9200 - val_loss: 0.2303
Saved /kaggle/working/results/figures_pdf/RQ2_confusion_matrix_bias_risks.pdf


,class,precision,recall,f1-score,support,error_rate
0,O,0.84,0.84,0.84,75.0,0.16
1,R,0.84,0.84,0.84,75.0,0.16
